In [8]:
import pandas as pd

In [9]:
# Load cleaned dataset
df = pd.read_csv("../data/raw/crime_dataset_india.csv")

In [10]:
# Normalize columns
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df["city"] = df["city"].astype(str).str.strip().str.lower()

In [11]:
# Count crimes per city
crime_counts = df.groupby("city").size().reset_index(name="crime_count")

crime_counts.sort_values(by="crime_count", ascending=False).head(10)

,city,crime_count
5,delhi,5400
17,mumbai,4415
2,bangalore,3588
8,hyderabad,2881
13,kolkata,2518
4,chennai,2493
21,pune,2212
1,ahmedabad,1817
10,jaipur,1479
14,lucknow,1456


In [12]:
def assign_hotspot(count):
    if count < 500:
        return "low"
    elif count < 1500:
        return "medium"
    else:
        return "high"

crime_counts["hotspot_level"] = crime_counts["crime_count"].apply(assign_hotspot)

In [13]:
crime_counts.head()

,city,crime_count,hotspot_level
0,agra,764,medium
1,ahmedabad,1817,high
2,bangalore,3588,high
3,bhopal,690,medium
4,chennai,2493,high


In [14]:
CITY_COORDINATES = {
    "delhi": (28.6139, 77.2090),
    "mumbai": (19.0760, 72.8777),
    "bangalore": (12.9716, 77.5946),
    "chennai": (13.0827, 80.2707),
    "kolkata": (22.5726, 88.3639),
    "pune": (18.5204, 73.8567),
    "hyderabad": (17.3850, 78.4867),
    "ahmedabad": (23.0225, 72.5714),
    "jaipur": (26.9124, 75.7873),
    "lucknow": (26.8467, 80.9462),
}

crime_counts["latitude"] = crime_counts["city"].map(
    lambda x: CITY_COORDINATES.get(x, (None, None))[0]
)
crime_counts["longitude"] = crime_counts["city"].map(
    lambda x: CITY_COORDINATES.get(x, (None, None))[1]
)

crime_counts.dropna(inplace=True)

crime_counts.head()


,city,crime_count,hotspot_level,latitude,longitude
1,ahmedabad,1817,high,23.0225,72.5714
2,bangalore,3588,high,12.9716,77.5946
4,chennai,2493,high,13.0827,80.2707
5,delhi,5400,high,28.6139,77.2090
8,hyderabad,2881,high,17.3850,78.4867


In [15]:
crime_counts.shape

(10, 5)

In [16]:
crime_counts.head()

,city,crime_count,hotspot_level,latitude,longitude
1,ahmedabad,1817,high,23.0225,72.5714
2,bangalore,3588,high,12.9716,77.5946
4,chennai,2493,high,13.0827,80.2707
5,delhi,5400,high,28.6139,77.2090
8,hyderabad,2881,high,17.3850,78.4867


# Model Training

In [17]:
# Select ML features
X = crime_counts[["crime_count", "latitude", "longitude"]]

In [18]:
from sklearn.preprocessing import StandardScaler

In [19]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

array([[-0.82304479,  0.39846595, -1.16943313],
       [ 0.62170922, -1.48877013, -0.04517016],
       [-0.2715746 , -1.46790912,  0.55377874],
       [ 2.09991038,  1.44835122, -0.13147288],
       [ 0.04494972, -0.66007541,  0.1544944 ]])

In [20]:
from sklearn.cluster import KMeans

In [21]:
# Train KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
crime_counts["cluster"] = kmeans.fit_predict(X_scaled)

crime_counts[["city", "crime_count", "cluster"]]

,city,crime_count,cluster
1,ahmedabad,1817,1
2,bangalore,3588,2
4,chennai,2493,2
5,delhi,5400,0
8,hyderabad,2881,2
10,jaipur,1479,1
13,kolkata,2518,2
14,lucknow,1456,1
17,mumbai,4415,1
21,pune,2212,1


In [22]:
# Analyze clusters
cluster_summary = crime_counts.groupby("cluster")["crime_count"].mean()
cluster_summary

cluster
0    5400.0
1    2275.8
2    2870.0
Name: crime_count, dtype: float64

In [23]:
# Map cluster to hotspot severity
cluster_map = {
    cluster_summary.idxmin(): "low",
    cluster_summary.idxmax(): "high"
}

In [24]:
# Remaining cluster = medium
remaining = set(cluster_summary.index) - set(cluster_map.keys())
cluster_map[list(remaining)[0]] = "medium"

crime_counts["ai_hotspot_level"] = crime_counts["cluster"].map(cluster_map)

crime_counts[["city", "crime_count", "ai_hotspot_level"]]

,city,crime_count,ai_hotspot_level
1,ahmedabad,1817,low
2,bangalore,3588,medium
4,chennai,2493,medium
5,delhi,5400,high
8,hyderabad,2881,medium
10,jaipur,1479,low
13,kolkata,2518,medium
14,lucknow,1456,low
17,mumbai,4415,low
21,pune,2212,low


In [25]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(kmeans, "../models/hotspot_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

print("Model and scaler saved successfully")


Model and scaler saved successfully


In [26]:
crime_counts[["city", "crime_count", "ai_hotspot_level"]]


,city,crime_count,ai_hotspot_level
1,ahmedabad,1817,low
2,bangalore,3588,medium
4,chennai,2493,medium
5,delhi,5400,high
8,hyderabad,2881,medium
10,jaipur,1479,low
13,kolkata,2518,medium
14,lucknow,1456,low
17,mumbai,4415,low
21,pune,2212,low


In [27]:
import os

# Ensure outputs folder exists
os.makedirs("../outputs", exist_ok=True)

# Save final AI results
crime_counts.to_csv("../outputs/crime_counts_with_ai_labels.csv", index=False)

print("crime_counts_with_ai_labels.csv saved successfully")


crime_counts_with_ai_labels.csv saved successfully


In [28]:
# doing prediction about the crime

In [29]:
import pandas as pd

df = pd.read_csv("../data/raw/crime_dataset_india.csv")

# Normalize columns
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df["city"] = df["city"].astype(str).str.strip().str.lower()

# Convert date column
df["date_reported"] = pd.to_datetime(df["date_reported"], errors="coerce")

df = df.dropna(subset=["date_reported"])

df.head()


,report_number,date_reported,date_of_occurrence,time_of_occurrence,city,crime_code,crime_description,victim_age,victim_gender,weapon_used,crime_domain,police_deployed,case_closed,date_case_closed
0,1,2020-02-01 00:00:00,01-01-2020 00:00,01-01-2020 01:11,ahmedabad,576,IDENTITY THEFT,16,M,Blunt Object,Violent Crime,13,No,NaN
1,2,2020-01-01 19:00:00,01-01-2020 01:00,01-01-2020 06:26,chennai,128,HOMICIDE,37,M,Poison,Other Crime,9,No,NaN
2,3,2020-02-01 05:00:00,01-01-2020 02:00,01-01-2020 14:30,ludhiana,271,KIDNAPPING,48,F,Blunt Object,Other Crime,15,No,NaN
3,4,2020-01-01 05:00:00,01-01-2020 03:00,01-01-2020 14:46,pune,170,BURGLARY,49,F,Firearm,Other Crime,1,Yes,29-04-2020 05:00
4,5,2020-01-01 21:00:00,01-01-2020 04:00,01-01-2020 16:51,pune,421,VANDALISM,30,F,Other,Other Crime,18,Yes,08-01-2020 21:00


In [30]:
df["year"] = df["date_reported"].dt.year
df["month"] = df["date_reported"].dt.month

df[["city", "year", "month"]].head()


,city,year,month
0,ahmedabad,2020,2
1,chennai,2020,1
2,ludhiana,2020,2
3,pune,2020,1
4,pune,2020,1


In [31]:
monthly_crime = (
    df.groupby(["city", "year", "month"])
      .size()
      .reset_index(name="monthly_crime_count")
)

monthly_crime.head()


,city,year,month,monthly_crime_count
0,agra,2020,1,6
1,agra,2020,2,4
2,agra,2020,3,8
3,agra,2020,4,4
4,agra,2020,5,3


In [32]:
import numpy as np

def get_trend(city_df):
    if len(city_df) < 2:
        return "stable"
    
    x = np.arange(len(city_df))
    y = city_df["monthly_crime_count"].values
    
    slope = np.polyfit(x, y, 1)[0]
    
    if slope > 0:
        return "increasing"
    elif slope < 0:
        return "decreasing"
    else:
        return "stable"


In [33]:
crime_trends = []

for city, group in monthly_crime.groupby("city"):
    group = group.sort_values(["year", "month"])
    trend = get_trend(group)
    
    crime_trends.append({
        "city": city,
        "crime_trend": trend
    })

crime_trends_df = pd.DataFrame(crime_trends)
crime_trends_df


,city,crime_trend
0,agra,decreasing
1,ahmedabad,decreasing
2,bangalore,decreasing
3,bhopal,decreasing
4,chennai,decreasing
5,delhi,decreasing
6,faridabad,decreasing
7,ghaziabad,decreasing
8,hyderabad,decreasing
9,indore,decreasing


In [34]:
# Recreate cluster -> label mapping safely
cluster_summary = crime_counts.groupby("cluster")["crime_count"].mean()

cluster_map = {
    str(cluster_summary.idxmin()): "low",
    str(cluster_summary.idxmax()): "high"
}

# Remaining cluster is medium
remaining = set(cluster_summary.index.astype(str)) - set(cluster_map.keys())
cluster_map[list(remaining)[0]] = "medium"

cluster_map


{'1': 'low', '0': 'high', '2': 'medium'}

In [35]:
import json
import os

os.makedirs("../models", exist_ok=True)

cluster_label_map = cluster_map  # from your training step

with open("../models/cluster_label_map.json", "w") as f:
    json.dump(cluster_label_map, f)

print("Cluster label map saved")


Cluster label map saved
